# Party axis — Gündüz replication + axis-rotation test (SDL 2026)

**Run 2026-07-16.** Motivation: Gündüz (2024, *SEBSS* 24(1)) fits per-wave binary logits on army
confidence in WVS Turkey 5/6/7 (same source data as our WV5–WV7) and finds CHP voters reversing
from high trust to distrust; he has no religiosity variable. We test whether the army's religious
gradient (dissolved by 2001, our headline) was replaced by a partisan one — axis rotation — and
validate our build against his published numbers. Terminology, used literally throughout:
**GD_ONLY** = "a great deal" only (his Table 2); **GD_PLUS** = "a great deal" + "quite a lot"
(his Table 3; = our binary "high confidence"). New files only: this notebook,
`results/party_axis_table.csv`.

## Verdicts

1. **GÜNDÜZ REPLICATION: CLOSE — effectively exact.** All six CHP cells match to ≤0.005 with identical SEs (GD_ONLY: +0.791 (0.221) / +0.204 (0.157) / −0.465 (0.186) vs his +0.792 (0.22) / +0.206 (0.157) / −0.466 (0.186); GD_PLUS: +0.656 (0.352) / +0.414 (0.187) / −0.545 (0.271) vs +0.656 (0.352) / +0.418 (0.187) / −0.55 (0.271)); pro-Kurdish −3.196 / −1.711 / −0.956 vs −3.193 / −1.708 / −0.958; N = 1,269 / 1,525 / 2,324 — **exact**. Side finding: the exact match proves his Table-2/3 party models contain **no left-right term** (his LR values come from separate model columns).
2. **AXIS ROTATION: SUPPORTED.** Joint model (2c), WV7 total gradients: **CHP = −0.389 (p = .005)**, **nr = +0.074 (p = .47)** — by 2018 the army carries a partisan gradient and no religious one. Army-specific: courts keeps the religious gradient (−0.413, p < .001) with no partisan one (+0.209, p = .12); police shows neither at WV7 (nr −0.182, p = .07; CHP +0.089, p = .52).
3. **SEPARABILITY:** corr(nr, CHP) = **+0.247** pooled (+0.21 to +0.28 per wave); condition number of the standardized interaction block = **5.9** (< 30 benign); VIFs ≤ 6.1. The two cleavages are statistically separable.
4. **IDEOLOGY: sign flip reproduces directionally.** 2018 positive and significant — LR = +0.193*** (GD_ONLY) / +0.166*** (GD_PLUS) vs his +0.205*** / +0.158*** → external replication, **not a WV7 coding artifact**. 2011 negative in sign on GD_PLUS (−0.037) but n.s. (p = .25) vs his −0.06**. In the joint model, **lr × WV7 = −0.003 (p = .93)**: the 2018 ideology gradient is absorbed by CHP × wave; CHP × WV7 survives (−1.763, p < .001).


In [1]:
# Setup + REPLICATION GATE (exact layer4b army spec, six waves)
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.tools.numdiff import approx_fprime, approx_hess
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
RESULTS = "results"
W567 = ["WV5", "WV6", "WV7"]
LATER6 = ["WV3", "WV4", "WV5", "WV6", "WV7"]

raw = pd.read_csv(f"{RESULTS}/pooled_extended_v2.csv")
raw["not_religious_dummy"] = raw["rel_person"].map(
    lambda v: 1 if v in (2, 3) else (0 if v == 1 else np.nan))

dfg = raw.dropna(subset=["cluster"]).copy()
for w in LATER6:
    dfg[f"wave_{w}"] = (dfg["wave"] == w).astype(int)
    dfg[f"nr_x_{w}"] = dfg["not_religious_dummy"] * dfg[f"wave_{w}"]
P6 = ["not_religious_dummy", "rel_attendance", "rel_importance_god", "left_right",
      "age", "female", "education_raw", "income_raw", "married_dummy",
      ] + [f"wave_{w}" for w in LATER6] + [f"nr_x_{w}" for w in LATER6]
sub = dfg[["conf_military"] + P6].dropna().copy()
sub["conf_military"] = sub["conf_military"].astype(int)
res_gate = OrderedModel(sub["conf_military"], sub[P6], distr="logit").fit(
    method="bfgs", maxiter=400, disp=False)
assert res_gate.mle_retvals.get("converged", False)
pub = pd.read_csv(f"{RESULTS}/layer4b_longwindow_interactions.csv")
prow = pub.query("institution=='conf_military' and predictor=='nr_x_WV4'").iloc[0]
g_c, g_s, g_n = float(res_gate.params["nr_x_WV4"]), float(res_gate.bse["nr_x_WV4"]), len(sub)
print(f"refit nr_x_WV4 = {g_c:+.6f} (SE {g_s:.6f}) n={g_n}  |  published "
      f"{float(prow['coef']):+.6f} ({float(prow['se']):.6f}) n={int(prow['n_obs'])}")
assert abs(g_c - prow["coef"]) < 1e-4 and abs(g_s - prow["se"]) < 1e-4, "GATE FAILED — STOP"
assert g_n == int(prow["n_obs"]) == 8915, "GATE FAILED on n — STOP"
print("REPLICATION GATE PASSED")

ROWS = []
def add(model, dv, sample, term, coef, se, n, p=None):
    z = coef / se if (se is not None and not np.isnan(se) and se != 0) else np.nan
    if p is None:
        p = 2 * (1 - stats.norm.cdf(abs(z))) if not np.isnan(z) else np.nan
    ROWS.append({"model": model, "dv_definition": dv, "sample": sample, "term": term,
                 "coef": coef, "se": se, "z": z, "p": p, "n": n})
add("GATE", "ordinal_1_4", "pooled_WV2-7", "nr_x_WV4", g_c, g_s, g_n)


refit nr_x_WV4 = +0.716155 (SE 0.182816) n=8915  |  published +0.716155 (0.182816) n=8915
REPLICATION GATE PASSED


## Step 0 — inventory (no models)

`party_recode` (from `pool_final.py`): WV2–WV4 are all `pre_AKP_era`; usable categories exist only
in WV5–WV7, with CHP separately identifiable in all three. **`party_recode` has no pro-Kurdish
category in WV5/WV6** — DTP (2007) and BDP (2011) were folded into `other`. They are separable
from `party_raw`: **DTP = 792049** (WV5, n = 69), **BDP = 792054** (WV6, n = 64), **HDP = 792005**
(WV7, n = 208) — labels verified against the official results books (WV5 results PDF p. 127:
"Democratic Society Party (DTP)" 4.9% weighted ≈ 69/1,346; WV6 results PDF p. 123: "BDP …
'Peace and Democracy Party'" 4.1% ≈ 64/1,605). MHP exists in all three waves; İYİ (IP) exists in
WV7 only (792050, n = 113; the same code in WV5 is Genç Parti — `pool_final.py` already handles
this). All party dummies below are built from `party_raw` codes; non-response (−1, −2, 2, 5)
stays in the reference category.

In [2]:
# 0a/0b — party coding per wave
print("party_recode x wave (raw n):")
print(pd.crosstab(raw["party_recode"].fillna("MISSING"), raw["wave"]).to_string())
PK_CODE = {"WV5": 792049, "WV6": 792054, "WV7": 792005}   # DTP / BDP / HDP
g0 = raw[raw["wave"].isin(W567)]
print("\nparty_raw codes, WV5-WV7 (n>=8):")
for w in W567:
    vc = g0.loc[g0["wave"] == w, "party_raw"].value_counts()
    print(f"  {w}:", {int(k): int(v) for k, v in vc.items() if v >= 8})
for w in W567:
    d = g0[g0["wave"] == w]
    for lbl, code in [("AKP", 792008), ("CHP", 792002), ("MHP", 792006),
                      ("pro_Kurdish", PK_CODE[w]), ("IYI", 792050 if w == "WV7" else None)]:
        if code is None:
            continue
        n_ = int((d["party_raw"] == code).sum())
        add("INVENTORY", "", w, f"n(party={lbl})", np.nan, np.nan, n_)

# 0c — nr x party overlap, WV5-WV7
def party_cat(row):
    if row["party_raw"] == 792008: return "AKP"
    if row["party_raw"] == 792002: return "CHP"
    if row["party_raw"] == 792006: return "MHP"
    if row["party_raw"] == PK_CODE[row["wave"]]: return "pro_Kurdish"
    if row["wave"] == "WV7" and row["party_raw"] == 792050: return "IYI"
    if pd.notna(row["party_raw"]) and row["party_raw"] > 100000: return "other"
    return "no_pref/missing"
g0 = g0.copy()
g0["party_cat"] = g0.apply(party_cat, axis=1)
for w in W567:
    d = g0[g0["wave"] == w]
    t = pd.crosstab(d["not_religious_dummy"].fillna("nr_missing"), d["party_cat"])
    print(f"\n0c — nr x party, {w}:")
    print(t.to_string())
    for nr_, r_ in t.iterrows():
        for pc, n_ in r_.items():
            add("INVENTORY", "", w, f"n(nr={nr_},party={pc})", np.nan, np.nan, int(n_))

# 0d — left_right availability
lr = raw.groupby("wave")["left_right"].agg(["count", "size"])
lr["pct_nonnull"] = (100 * lr["count"] / lr["size"]).round(1)
print("\n0d — left_right non-null per wave:")
print(lr.to_string())
for w_, r_ in lr.iterrows():
    add("INVENTORY", "", w_, "n(left_right nonnull)", np.nan, np.nan, int(r_["count"]))


party_recode x wave (raw n):
wave           WV2   WV3   WV4  WV5  WV6   WV7
party_recode                                  
AKP              0     0     0  451  649  1110
CHP              0     0     0  160  395   519
HDP              0     0     0    0    0   208
IYI              0     0     0    0    0   113
MHP              0     0     0   88  123   261
MISSING          0     0     0  493  374   196
other            0     0     0  154   64     8
pre_AKP_era   1030  1907  3401    0    0     0

party_raw codes, WV5-WV7 (n>=8):
  WV5: {792008: 451, -1: 205, 2: 178, 792002: 160, -2: 94, 792006: 88, 792049: 69, 792004: 24, 792050: 23, 5: 16, 792001: 14, 792051: 11, 792003: 11}
  WV6: {792008: 649, 792002: 395, -2: 240, 792006: 123, 2: 97, 792054: 64, 5: 26, -1: 11}
  WV7: {792008: 1110, 792002: 519, 792006: 261, 792005: 208, -2: 180, 792050: 113, 5: 16, 792051: 8}

0c — nr x party, WV5:
party_cat            AKP  CHP  MHP  no_pref/missing  other  pro_Kurdish
not_religious_dummy            

## Step 1 — Gündüz replication (GUNDUZ_SPEC)

His spec, approximated: per-wave binary logit, DV GD_ONLY then GD_PLUS; party dummies AKP, CHP,
MHP, pro-Kurdish (+ İYİ in WV7), party non-response in the reference; controls: confidence in
parliament (1–4, higher = more confident), life satisfaction, education, married, age, male;
listwise on DV + controls (party never restricts the sample). **This includes his parliament
control, which our own models deliberately do not use.** No left-right term — the exact
reproduction below shows his Models 1/3/5 cannot contain one; LR is examined in step 3.
Comparison criterion: CLOSE = same sign, same .05-significance class (his class derived from his
published coef/SE — his star convention marks p<.10, see the note under the table), and
|Δcoef| < 2×our SE.

In [3]:
# GUNDUZ_SPEC fits, both DVs x three waves
g = raw[raw["wave"].isin(W567)].copy()
g["male"] = 1 - g["female"]
g["AKP_dummy"] = (g["party_raw"] == 792008).astype(int)
g["CHP_dummy"] = (g["party_raw"] == 792002).astype(int)
g["MHP_dummy"] = (g["party_raw"] == 792006).astype(int)
g["PK_dummy"] = (g["wave"].map(PK_CODE) == g["party_raw"]).astype(int)
g["IYI_dummy"] = ((g["wave"] == "WV7") & (g["party_raw"] == 792050)).astype(int)
g["GD_ONLY"] = np.where(g["conf_military"].isna(), np.nan, (g["conf_military"] == 4).astype(float))
g["GD_PLUS"] = np.where(g["conf_military"].isna(), np.nan, (g["conf_military"] >= 3).astype(float))
CTRL_G = ["conf_parliament", "life_satisfaction", "education_raw", "married_dummy", "age", "male"]

def gunduz_fit(w, dv, extra=None):
    d = g[g["wave"] == w]
    parties = ["AKP_dummy", "CHP_dummy", "MHP_dummy", "PK_dummy"] + (
        ["IYI_dummy"] if w == "WV7" else [])
    preds = parties + CTRL_G + (extra or [])
    d = d[[dv] + preds].dropna()
    keep = [c for c in preds if d[c].nunique() > 1]
    r = sm.Logit(d[dv].astype(int), sm.add_constant(d[keep])).fit(disp=False, maxiter=400)
    assert r.mle_retvals["converged"], f"{w} {dv}"
    return r, len(d)

HIS = {  # published anchors: (coef, se or None). His stars mark p<.10 as * (see note below),
    # so his .05-significance class is derived from his own coef/SE, not his stars.
    ("GD_ONLY", "WV5", "CHP_dummy"): (0.792, 0.22),
    ("GD_ONLY", "WV6", "CHP_dummy"): (0.206, 0.157),
    ("GD_ONLY", "WV7", "CHP_dummy"): (-0.466, 0.186),
    ("GD_PLUS", "WV5", "CHP_dummy"): (0.656, 0.352),
    ("GD_PLUS", "WV6", "CHP_dummy"): (0.418, 0.187),
    ("GD_PLUS", "WV7", "CHP_dummy"): (-0.55, 0.271),
    ("GD_ONLY", "WV5", "PK_dummy"): (-3.193, None),
    ("GD_ONLY", "WV6", "PK_dummy"): (-1.708, None),
    ("GD_ONLY", "WV7", "PK_dummy"): (-0.958, None),
}
HIS_N = {"WV5": 1269, "WV6": 1525, "WV7": 2324}

gfits, cmp_rows = {}, []
for dv in ["GD_ONLY", "GD_PLUS"]:
    for w in W567:
        r, n = gunduz_fit(w, dv)
        gfits[(dv, w)] = (r, n)
        for t in [c for c in r.params.index if c != "const"]:
            add("GUNDUZ_SPEC", dv, w, t, float(r.params[t]), float(r.bse[t]), n)
        for t in ["CHP_dummy", "PK_dummy"]:
            if (dv, w, t) not in HIS:
                continue
            hc, hs = HIS[(dv, w, t)]
            hsig = (abs(hc / hs) > 1.959964) if hs else None
            oc, os_, op = float(r.params[t]), float(r.bse[t]), float(r.pvalues[t])
            close = (np.sign(oc) == np.sign(hc)
                     and (hsig is None or (op < .05) == hsig)
                     and abs(oc - hc) < 2 * os_)
            cmp_rows.append({"dv": dv, "wave": w, "term": t,
                             "ours": f"{oc:+.4f} ({os_:.4f}) p={op:.4f}",
                             "his": f"{hc:+.3f}" + (f" ({hs})" if hs else ""),
                             "diff": round(abs(oc - hc), 4),
                             "verdict": "CLOSE" if close else "DIVERGENT",
                             "n_ours": n, "n_his": HIS_N[w]})
cmp = pd.DataFrame(cmp_rows)
print(cmp.to_string(index=False))
print("\nNote: GD_PLUS WV5 CHP two-sided p = "
      f"{float(gfits[('GD_PLUS','WV5')][0].pvalues['CHP_dummy']):.4f} — his '*' evidently marks "
      "p<.10 (or one-tailed); coefficient and SE match exactly. His '**' on GD_ONLY WV7 "
      f"(our p = {float(gfits[('GD_ONLY','WV7')][0].pvalues['CHP_dummy']):.4f}) likewise implies "
      "a laxer star convention; .05-class agreement holds in every cell.")


     dv wave      term                      ours            his   diff verdict  n_ours  n_his
GD_ONLY  WV5 CHP_dummy +0.7913 (0.2208) p=0.0003  +0.792 (0.22) 0.0007   CLOSE    1269   1269
GD_ONLY  WV5  PK_dummy -3.1961 (0.5346) p=0.0000         -3.193 0.0031   CLOSE    1269   1269
GD_ONLY  WV6 CHP_dummy +0.2036 (0.1574) p=0.1957 +0.206 (0.157) 0.0024   CLOSE    1525   1525
GD_ONLY  WV6  PK_dummy -1.7114 (0.4887) p=0.0005         -1.708 0.0034   CLOSE    1525   1525
GD_ONLY  WV7 CHP_dummy -0.4647 (0.1863) p=0.0126 -0.466 (0.186) 0.0013   CLOSE    2324   2324
GD_ONLY  WV7  PK_dummy -0.9564 (0.2455) p=0.0001         -0.958 0.0016   CLOSE    2324   2324
GD_PLUS  WV5 CHP_dummy +0.6560 (0.3516) p=0.0621 +0.656 (0.352) 0.0000   CLOSE    1269   1269
GD_PLUS  WV6 CHP_dummy +0.4135 (0.1868) p=0.0268 +0.418 (0.187) 0.0045   CLOSE    1525   1525
GD_PLUS  WV7 CHP_dummy -0.5445 (0.2712) p=0.0447 -0.550 (0.271) 0.0055   CLOSE    2324   2324

Note: GD_PLUS WV5 CHP two-sided p = 0.0621 — his '*' eviden

## Step 2 — axis rotation (our specification)

Pooled WV5–WV7, WV5 baseline, ordinal logit on `conf_military`, layer4b controls (incl.
`rel_attendance`), cluster-non-missing sample as in layer4b. **No parliament-confidence control:
in a hybrid regime, parliament confidence is downstream of the partisan cleavage being estimated,
so controlling for it conditions on a collider-like quantity.** Party dummies from `party_raw`
(AKP, CHP, MHP, pro-Kurdish DTP/BDP/HDP, other-stated-party; non-response = reference; İYİ counted
in `other`). Three nested models on the **same sample** (listwise on outcome + controls + nr;
party dummies never missing): 2a nr × wave only; 2b CHP × wave only (other party dummies as main
effects); 2c joint. Totals = main + interaction, delta-method SE (layer4b `wave_total` approach).

In [4]:
# 2a/2b/2c on the same sample
d2 = raw[raw["wave"].isin(W567)].dropna(subset=["cluster"]).copy()
d2["AKP_dummy"] = (d2["party_raw"] == 792008).astype(int)
d2["CHP_dummy"] = (d2["party_raw"] == 792002).astype(int)
d2["MHP_dummy"] = (d2["party_raw"] == 792006).astype(int)
d2["PK_dummy"] = (d2["wave"].map(PK_CODE) == d2["party_raw"]).astype(int)
stated = d2["party_raw"].notna() & (d2["party_raw"] > 100000)
d2["other_dummy"] = (stated & (d2[["AKP_dummy", "CHP_dummy", "MHP_dummy", "PK_dummy"]
                                  ].sum(axis=1) == 0)).astype(int)
d2["wave_WV6"] = (d2["wave"] == "WV6").astype(int)
d2["wave_WV7"] = (d2["wave"] == "WV7").astype(int)
for a, b in [("nr", "not_religious_dummy"), ("chp", "CHP_dummy"), ("lr", "left_right")]:
    d2[f"{a}_x_wv6"] = d2[b] * d2["wave_WV6"]
    d2[f"{a}_x_wv7"] = d2[b] * d2["wave_WV7"]

CTRL2 = ["rel_attendance", "rel_importance_god", "left_right",
         "age", "female", "education_raw", "income_raw", "married_dummy"]
PARTY5 = ["AKP_dummy", "CHP_dummy", "MHP_dummy", "PK_dummy", "other_dummy"]
WVd = ["wave_WV6", "wave_WV7"]
NRB = ["not_religious_dummy", "nr_x_wv6", "nr_x_wv7"]
CHB = ["chp_x_wv6", "chp_x_wv7"]

def fit_ord(frame, outcome, preds, start=None):
    s = frame[[outcome] + preds].dropna().copy()
    s[outcome] = s[outcome].astype(int)
    keep = [p for p in preds if s[p].nunique() > 1]
    r = OrderedModel(s[outcome], s[keep], distr="logit").fit(
        method="bfgs", maxiter=400, disp=False, start_params=start)
    assert r.mle_retvals.get("converged", False), outcome
    return r, len(s)

def total(res, main, inter, cov=None, params=None):
    L = np.zeros(len(res.params))
    L[res.params.index.get_loc(main)] = 1
    if inter:
        L[res.params.index.get_loc(inter)] = 1
    pv = np.asarray(res.params, float) if params is None else params
    C = res.cov_params() if cov is None else cov
    c = float(L @ pv)
    se = float(np.sqrt(L @ C @ L))
    return c, se, 2 * (1 - stats.norm.cdf(abs(c / se)))

SAMP_VARS = ["conf_military"] + CTRL2 + ["not_religious_dummy"]
s2 = d2.dropna(subset=SAMP_VARS)
print(f"step-2 sample n = {len(s2)}  by wave: {dict(s2['wave'].value_counts().reindex(W567))}")
print("party n in sample:", {c: int(s2[c].sum()) for c in PARTY5})

M2a, n2 = fit_ord(s2, "conf_military", CTRL2 + WVd + NRB)
M2b, _ = fit_ord(s2, "conf_military", CTRL2 + WVd + PARTY5 + CHB)
M2c, _ = fit_ord(s2, "conf_military", CTRL2 + WVd + NRB + PARTY5 + CHB)
KEY = ["not_religious_dummy", "nr_x_wv6", "nr_x_wv7", "CHP_dummy", "chp_x_wv6", "chp_x_wv7"]
side = []
for t in KEY:
    row = {"term": t}
    for lbl, r in [("2a_nr_only", M2a), ("2b_chp_only", M2b), ("2c_joint", M2c)]:
        row[lbl] = (f"{float(r.params[t]):+.4f} ({float(r.bse[t]):.4f}) "
                    f"p={float(r.pvalues[t]):.4g}") if t in r.params.index else "—"
    side.append(row)
print("\n", pd.DataFrame(side).to_string(index=False))
for lbl, r, preds in [("OURS_2a", M2a, CTRL2 + WVd + NRB),
                      ("OURS_2b", M2b, CTRL2 + WVd + PARTY5 + CHB),
                      ("OURS_2c", M2c, CTRL2 + WVd + NRB + PARTY5 + CHB)]:
    for t in preds:
        if t in r.params.index:
            add(lbl, "ordinal_1_4", "pooled_WV5-7", t,
                float(r.params[t]), float(r.bse[t]), n2)
print("\n2c totals (main + interaction, delta SE):")
for lbl, m, i6, i7 in [("nr", "not_religious_dummy", "nr_x_wv6", "nr_x_wv7"),
                       ("CHP", "CHP_dummy", "chp_x_wv6", "chp_x_wv7")]:
    for wl, inter in [("WV5", None), ("WV6", i6), ("WV7", i7)]:
        c, se, p = total(M2c, m, inter)
        print(f"  {lbl} total @{wl}: {c:+.4f} ({se:.4f}) p={p:.4g}")
        add("OURS_2c", "ordinal_1_4", "pooled_WV5-7", f"{lbl}_total_{wl}", c, se, n2, p=p)


step-2 sample n = 4221  by wave: {'WV5': 1064, 'WV6': 1327, 'WV7': 1830}
party n in sample: {'AKP_dummy': 1834, 'CHP_dummy': 898, 'MHP_dummy': 416, 'PK_dummy': 252, 'other_dummy': 163}



                term                2a_nr_only                  2b_chp_only                     2c_joint
not_religious_dummy -0.0532 (0.1670) p=0.7502                            —    -0.2246 (0.1725) p=0.1928
           nr_x_wv6 -0.2483 (0.2134) p=0.2446                            —    -0.2099 (0.2208) p=0.3418
           nr_x_wv7 +0.0126 (0.1892) p=0.9471                            —      +0.2985 (0.1971) p=0.13
          CHP_dummy                         — +1.2957 (0.2266) p=1.073e-08 +1.3465 (0.2340) p=8.733e-09
          chp_x_wv6                         —  -0.8014 (0.2473) p=0.001192  -0.8016 (0.2570) p=0.001817
          chp_x_wv7                         — -1.6388 (0.2409) p=1.022e-11 -1.7353 (0.2509) p=4.653e-12

2c totals (main + interaction, delta SE):
  nr total @WV5: -0.2246 (0.1725) p=0.1928
  nr total @WV6: -0.4345 (0.1448) p=0.002692
  nr total @WV7: +0.0739 (0.1030) p=0.473
  CHP total @WV5: +1.3465 (0.2340) p=8.733e-09
  CHP total @WV6: +0.5449 (0.1429) p=0.000137
  CH

In [5]:
# 2d — collinearity (education_defense.ipynb approach); 2e — police + courts
cc = s2[["not_religious_dummy", "CHP_dummy"]]
print(f"corr(nr, CHP) pooled: {cc.corr().iloc[0, 1]:+.4f} (n={len(cc)});  per wave:",
      {w: round(float(s2[s2['wave'] == w][['not_religious_dummy', 'CHP_dummy']]
                 .corr().iloc[0, 1]), 4) for w in W567})
X = s2[["not_religious_dummy", "CHP_dummy", "nr_x_wv6", "nr_x_wv7", "chp_x_wv6", "chp_x_wv7"]]
Xs = ((X - X.mean()) / X.std()).values
cond = np.linalg.cond(Xs)
vif = pd.Series(np.diag(np.linalg.inv(X.corr().values)), index=X.columns).round(2)
print(f"condition number (standardized interaction block): {cond:.1f}  (< 30 = benign)")
print("VIF:", dict(vif))
add("OURS_2d", "", "pooled_WV5-7", "corr(nr,CHP)", float(cc.corr().iloc[0, 1]), np.nan, len(cc))
add("OURS_2d", "", "pooled_WV5-7", "condition_number", float(cond), np.nan, len(s2))

for inst, tag in [("conf_police", "OURS_2c_police"), ("conf_courts", "OURS_2c_courts")]:
    si = d2.dropna(subset=[inst] + CTRL2 + ["not_religious_dummy"])
    ri, ni = fit_ord(si, inst, CTRL2 + WVd + NRB + PARTY5 + CHB)
    print(f"\n2e — {inst} (n={ni}):")
    for t in KEY:
        print(f"  {t:<22} {float(ri.params[t]):+.4f} ({float(ri.bse[t]):.4f}) "
              f"p={float(ri.pvalues[t]):.4g}")
        add(tag, "ordinal_1_4", "pooled_WV5-7", t, float(ri.params[t]), float(ri.bse[t]), ni)
    for lbl, m, i7 in [("nr", "not_religious_dummy", "nr_x_wv7"),
                       ("CHP", "CHP_dummy", "chp_x_wv7")]:
        c, se, p = total(ri, m, i7)
        print(f"  total {lbl}@WV7: {c:+.4f} ({se:.4f}) p={p:.4g}")
        add(tag, "ordinal_1_4", "pooled_WV5-7", f"{lbl}_total_WV7", c, se, ni, p=p)


corr(nr, CHP) pooled: +0.2470 (n=4221);  per wave: {'WV5': 0.2754, 'WV6': 0.2068, 'WV7': 0.2636}
condition number (standardized interaction block): 5.9  (< 30 = benign)
VIF: {'not_religious_dummy': 4.53, 'CHP_dummy': 6.12, 'nr_x_wv6': 2.36, 'nr_x_wv7': 3.84, 'chp_x_wv6': 3.69, 'chp_x_wv7': 4.4}



2e — conf_police (n=4223):
  not_religious_dummy    -0.6922 (0.1560) p=9.102e-06
  nr_x_wv6               -0.0447 (0.2066) p=0.8287
  nr_x_wv7               +0.5100 (0.1820) p=0.005073
  CHP_dummy              +0.5128 (0.1854) p=0.005664
  chp_x_wv6              -0.4651 (0.2142) p=0.02993
  chp_x_wv7              -0.4243 (0.2068) p=0.04022
  total nr@WV7: -0.1822 (0.1010) p=0.07133
  total CHP@WV7: +0.0885 (0.1364) p=0.5164



2e — conf_courts (n=4215):
  not_religious_dummy    -0.3326 (0.1599) p=0.03756
  nr_x_wv6               -0.1129 (0.2101) p=0.5911
  nr_x_wv7               -0.0804 (0.1842) p=0.6625
  CHP_dummy              +0.1415 (0.1873) p=0.4501
  chp_x_wv6              -0.1273 (0.2154) p=0.5545
  chp_x_wv7              +0.0674 (0.2081) p=0.7459
  total nr@WV7: -0.4130 (0.0981) p=2.564e-05
  total CHP@WV7: +0.2089 (0.1358) p=0.1238


## Step 3 — ideology

**3a** Left-right per wave in Gündüz-style models, both DVs. His party models provably exclude LR
(step 1), so his LR values (+0.205***/+0.158*** in 2018; +0.018 n.s. 2007; −0.06** GD_PLUS 2011)
come from separate model columns; we report LR two ways — added to GUNDUZ_SPEC
(`GUNDUZ_SPEC_LR`) and replacing the party block (`GUNDUZ_SPEC_LRONLY`) — both listwise on LR
(our release has no LR imputation; n drops accordingly). **3b** adds lr × wave to the joint
model 2c.

In [6]:
# 3a — LR per wave, both DVs, two variants
HIS_LR = {("GD_ONLY", "WV5"): "+0.018 n.s.", ("GD_ONLY", "WV7"): "+0.205***",
          ("GD_PLUS", "WV6"): "-0.06**", ("GD_PLUS", "WV7"): "+0.158***"}
rows3 = []
for dv in ["GD_ONLY", "GD_PLUS"]:
    for w in W567:
        r_lr, n_lr = gunduz_fit(w, dv, extra=["left_right"])
        d = g[g["wave"] == w][[dv] + CTRL_G + ["left_right"]].dropna()
        keep = [c for c in CTRL_G + ["left_right"] if d[c].nunique() > 1]
        r_only = sm.Logit(d[dv].astype(int), sm.add_constant(d[keep])).fit(
            disp=False, maxiter=400)
        assert r_only.mle_retvals["converged"]
        rows3.append({"dv": dv, "wave": w, "n": n_lr,
                      "LR (spec+parties)": f"{float(r_lr.params['left_right']):+.4f} "
                      f"({float(r_lr.bse['left_right']):.4f}) p={float(r_lr.pvalues['left_right']):.4f}",
                      "LR (no parties)": f"{float(r_only.params['left_right']):+.4f} "
                      f"({float(r_only.bse['left_right']):.4f}) p={float(r_only.pvalues['left_right']):.4f}",
                      "his": HIS_LR.get((dv, w), "—")})
        add("GUNDUZ_SPEC_LR", dv, w, "left_right",
            float(r_lr.params["left_right"]), float(r_lr.bse["left_right"]), n_lr)
        add("GUNDUZ_SPEC_LRONLY", dv, w, "left_right",
            float(r_only.params["left_right"]), float(r_only.bse["left_right"]), len(d))
print(pd.DataFrame(rows3).to_string(index=False))

# 3b — joint model + lr x wave
M3b, n3 = fit_ord(s2, "conf_military",
                  CTRL2 + WVd + NRB + PARTY5 + CHB + ["lr_x_wv6", "lr_x_wv7"])
print(f"\n3b — 2c + lr x wave (n={n3}):")
for t in ["not_religious_dummy", "nr_x_wv7", "CHP_dummy", "chp_x_wv7",
          "left_right", "lr_x_wv6", "lr_x_wv7"]:
    print(f"  {t:<22} {float(M3b.params[t]):+.4f} ({float(M3b.bse[t]):.4f}) "
          f"p={float(M3b.pvalues[t]):.4g}")
    add("OURS_3b", "ordinal_1_4", "pooled_WV5-7", t,
        float(M3b.params[t]), float(M3b.bse[t]), n3)
for lbl, m, i7 in [("nr", "not_religious_dummy", "nr_x_wv7"),
                   ("CHP", "CHP_dummy", "chp_x_wv7")]:
    c, se, p = total(M3b, m, i7)
    print(f"  total {lbl}@WV7: {c:+.4f} ({se:.4f}) p={p:.4g}")
    add("OURS_3b", "ordinal_1_4", "pooled_WV5-7", f"{lbl}_total_WV7", c, se, n3, p=p)


     dv wave    n         LR (spec+parties)           LR (no parties)         his
GD_ONLY  WV5 1095 +0.0593 (0.0325) p=0.0682 +0.0727 (0.0261) p=0.0053 +0.018 n.s.
GD_ONLY  WV6 1396 +0.0014 (0.0274) p=0.9584 +0.0201 (0.0240) p=0.4023           —
GD_ONLY  WV7 2088 +0.1929 (0.0251) p=0.0000 +0.2216 (0.0209) p=0.0000   +0.205***
GD_PLUS  WV5 1095 +0.1132 (0.0448) p=0.0116 +0.1765 (0.0353) p=0.0000           —
GD_PLUS  WV6 1396 -0.0370 (0.0322) p=0.2497 -0.0306 (0.0275) p=0.2653     -0.06**
GD_PLUS  WV7 2088 +0.1664 (0.0341) p=0.0000 +0.1829 (0.0271) p=0.0000   +0.158***



3b — 2c + lr x wave (n=4221):
  not_religious_dummy    -0.1751 (0.1753) p=0.3179
  nr_x_wv7               +0.2583 (0.2002) p=0.1969
  CHP_dummy              +1.4735 (0.2421) p=1.149e-09
  chp_x_wv7              -1.7633 (0.2665) p=3.692e-11
  left_right             +0.1423 (0.0282) p=4.568e-07
  lr_x_wv6               -0.1264 (0.0365) p=0.0005363
  lr_x_wv7               -0.0028 (0.0338) p=0.934
  total nr@WV7: +0.0832 (0.1034) p=0.4212
  total CHP@WV7: -0.2897 (0.1455) p=0.04643


## Step 4 — weighted sensitivity

Positional per-wave merge validated in `preflight_refits.ipynb` (100% row-for-row agreement):
V259 (WV5), V258 (WV6), W_WEIGHT ≡ 1.0 (WV7). Estimator as there: pseudo-ML ordered logit with
weights in the log-likelihood, Huber–White sandwich SEs; no clustering adjustment (no PSU ids in
WV5/WV6; WV7's weight is constant).

In [7]:
# weighted 2c
FILES = {"WV5": "F00007873-WV5_Data_Turkey_Excel_v20201117.1.xlsx",
         "WV6": "F00007629-WV6_Data_Turkey_2011_Excel_v20180912.xlsx",
         "WV7": "F00013222-WVS_Wave_7_Turkey_Excel_v5.1.xlsx"}
WCODE = {"WV5": "V259", "WV6": "V258", "WV7": "W_WEIGHT"}
parts = []
for w in W567:
    hm = {str(c).split(":", 1)[0].strip(): c
          for c in pd.read_excel(FILES[w], nrows=0).columns}
    src = pd.read_excel(FILES[w], usecols=[hm[WCODE[w]]]).rename(
        columns={hm[WCODE[w]]: "cal_weight"})
    src["wave"] = w
    src["row_in_wave"] = np.arange(len(src))
    parts.append(src)
d2w = d2.merge(pd.concat(parts, ignore_index=True), on=["wave", "row_in_wave"], how="left")
s2w = d2w.dropna(subset=SAMP_VARS)
assert len(s2w) == len(s2) and s2w["cal_weight"].notna().all()
dist = s2w.groupby("wave")["cal_weight"].agg(["count", "min", "max", "mean", "std"])
dist["cv"] = dist["std"] / dist["mean"]
print("analytic-sample weight distribution (weight variable per wave: "
      "WV5 V259, WV6 V258, WV7 W_WEIGHT=1):")
print(dist.round(4).to_string())
for w_, r_ in dist.iterrows():
    add("OURS_2c_WEIGHTED", "", w_, "weight_cv", float(r_["cv"]), np.nan, int(r_["count"]))

class WeightedOrderedModel(OrderedModel):
    def __init__(self, endog, exog, weights=None, **kw):
        super().__init__(endog, exog, **kw)
        self._w = np.asarray(weights, dtype=float)
    def loglikeobs(self, params):
        return self._w * super().loglikeobs(params)

preds_2c = CTRL2 + WVd + NRB + PARTY5 + CHB
sW = s2w[["conf_military"] + preds_2c + ["cal_weight"]].dropna(
    subset=["conf_military"] + preds_2c).copy()
sW["conf_military"] = sW["conf_military"].astype(int)
mw = WeightedOrderedModel(sW["conf_military"], sW[preds_2c],
                          weights=sW["cal_weight"].to_numpy(), distr="logit")
rw = mw.fit(method="bfgs", maxiter=400, disp=False, start_params=M2c.params.values)
assert rw.mle_retvals.get("converged", False)
pv = np.asarray(rw.params, float)
S = approx_fprime(pv, mw.loglikeobs)
H = approx_hess(pv, lambda q: mw.loglikeobs(q).sum())
Ainv = np.linalg.inv(-H)
covW = Ainv @ (S.T @ S) @ Ainv
seW = pd.Series(np.sqrt(np.diag(covW)), index=rw.params.index)
print(f"\nweighted 2c (n={len(sW)}), sandwich SEs — unweighted side by side:")
for t in KEY:
    wc, ws = float(rw.params[t]), float(seW[t])
    wp = 2 * (1 - stats.norm.cdf(abs(wc / ws)))
    print(f"  {t:<22} wtd {wc:+.4f} ({ws:.4f}) p={wp:.4g}   "
          f"unw {float(M2c.params[t]):+.4f} ({float(M2c.bse[t]):.4f})")
    add("OURS_2c_WEIGHTED", "ordinal_1_4", "pooled_WV5-7", t, wc, ws, len(sW))
WTOT = {}
for lbl, m, i7 in [("nr", "not_religious_dummy", "nr_x_wv7"),
                   ("CHP", "CHP_dummy", "chp_x_wv7")]:
    c, se, p = total(rw, m, i7, cov=covW, params=pv)
    WTOT[lbl] = (c, se, p)
    print(f"  weighted total {lbl}@WV7: {c:+.4f} ({se:.4f}) p={p:.4g}")
    add("OURS_2c_WEIGHTED", "ordinal_1_4", "pooled_WV5-7", f"{lbl}_total_WV7",
        c, se, len(sW), p=p)


analytic-sample weight distribution (weight variable per wave: WV5 V259, WV6 V258, WV7 W_WEIGHT=1):
      count     min     max    mean     std      cv
wave                                               
WV5    1064  0.7014  1.7313  0.9965  0.2459  0.2468
WV6    1327  0.6560  2.1132  0.9945  0.3533  0.3553
WV7    1830  1.0000  1.0000  1.0000  0.0000  0.0000



weighted 2c (n=4221), sandwich SEs — unweighted side by side:
  not_religious_dummy    wtd -0.2891 (0.1804) p=0.1089   unw -0.2246 (0.1725)
  nr_x_wv6               wtd -0.1172 (0.2328) p=0.6148   unw -0.2099 (0.2208)
  nr_x_wv7               wtd +0.3631 (0.1992) p=0.0684   unw +0.2985 (0.1971)
  CHP_dummy              wtd +1.4406 (0.2637) p=4.664e-08   unw +1.3465 (0.2340)
  chp_x_wv6              wtd -0.8471 (0.2857) p=0.003026   unw -0.8016 (0.2570)
  chp_x_wv7              wtd -1.7870 (0.2718) p=4.895e-11   unw -1.7353 (0.2509)
  weighted total nr@WV7: +0.0739 (0.0942) p=0.4323
  weighted total CHP@WV7: -0.3464 (0.1341) p=0.009798


## Summary — conf_military at WV7: religious vs partisan gradient

In [8]:
# final summary + tidy CSV
summ = []
for lbl, m, i7 in [("religious (nr)", "not_religious_dummy", "nr_x_wv7"),
                   ("partisan (CHP)", "CHP_dummy", "chp_x_wv7")]:
    c, se, p = total(M2c, m, i7)
    cw, sew_, pw = WTOT["nr" if "nr" in lbl else "CHP"]
    summ.append({"gradient at WV7 (2018)": lbl,
                 "unweighted": f"{c:+.4f} ({se:.4f}) p={p:.4g}",
                 "weighted": f"{cw:+.4f} ({sew_:.4f}) p={pw:.4g}"})
summary = pd.DataFrame(summ)
master = pd.DataFrame(ROWS, columns=["model", "dv_definition", "sample", "term",
                                     "coef", "se", "z", "p", "n"])
master.to_csv(f"{RESULTS}/party_axis_table.csv", index=False)
print(f"wrote results/party_axis_table.csv ({len(master)} rows)\n")
summary


wrote results/party_axis_table.csv (242 rows)



,gradient at WV7 (2018),unweighted,weighted
0,religious (nr),+0.0739 (0.1030) p=0.473,+0.0739 (0.0942) p=0.4323
1,partisan (CHP),-0.3887 (0.1389) p=0.005116,-0.3464 (0.1341) p=0.009798
